# Dask.Net - Dask for .NET

Parallel and distributed computing over the Gravicode.Science stack: lazy task graphs, chunked
arrays, partitioned DataFrames with a query optimiser, SQL, and a cluster of workers that recovers
from losing one.

*Dibuat oleh Gravicode Studios, dipimpin oleh Kang Fadhil*

Build the solution in Release first (`dotnet build Gravicode.Science.sln -c Release`).

In [ ]:
#r "../src/GraviNum/bin/Release/net10.0/Gravicode.Science.GraviNum.dll"
#r "../src/GraviFrame/bin/Release/net10.0/Gravicode.Science.GraviFrame.dll"
#r "../src/DaskNet/bin/Release/net10.0/Gravicode.Science.DaskNet.dll"

using Gravicode.Science.GraviNum;
using Gravicode.Science.GraviFrame;
using Gravicode.Science.DaskNet;
using Gravicode.Science.DaskNet.Distributed;
using Gravicode.Science.DaskNet.Sql;
using static Gravicode.Science.DaskNet.Expr;

Console.WriteLine(DaskNetInfo.Banner());

## Delayed: building a graph lazily (`dask.delayed`)

Nothing runs until `Compute`. Shared ancestors run once, and chains of single-input tasks are
fused into one task before scheduling.

In [ ]:
var raw = Enumerable.Range(0, 4).Select(i => Dask.Delayed((int part) => Enumerable.Range(part * 100, 100).Select(v => (double)v).ToArray(), Dask.Value(i))).ToList();
var cleaned = raw.Select(r => r.Then(values => values.Where(v => v % 3 != 0).ToArray())).ToList();
var totalSum = Dask.All(cleaned).Then(parts => parts.Sum(p => p.Sum()));

Console.WriteLine($"sum = {totalSum.Compute()}");
Console.WriteLine($"tasks before fusion: {TaskGraph.FromCollections(totalSum).Count}, after: {Dask.Graph(totalSum).Count}");
Console.WriteLine(Dask.Graph(totalSum).ToMermaid());

## Bag: semi-structured data (`dask.bag`)

A bag is a partitioned sequence. Reductions combine per-partition results in a tree, and
`FoldBy`/`Frequencies` aggregate before shuffling, so only one row per key per partition moves.

In [ ]:
var reviews = Bag.ReadText("../datasets/imdb_reviews.csv", blockSize: 4 * 1024);
var lengths = reviews.Map(line => (double)line.Length);
Console.WriteLine($"{reviews.PartitionCount} blocks, {lengths.Count().Compute()} lines, mean length {lengths.Mean(v => v).Compute():F1}");

var longWords = reviews.FlatMap(line => line.ToLowerInvariant().Split(' ', StringSplitOptions.RemoveEmptyEntries)).Filter(w => w.Length > 6);
foreach (var (word, count) in longWords.Frequencies().Compute().OrderByDescending(f => f.Count).Take(5))
    Console.WriteLine($"  {word,-14} {count}");

## Array: chunked NdArrays (`dask.array`)

Each chunk is a GraviNum `NdArray`. Reductions are computed per chunk and combined in a tree;
variance uses Chan's merge of (count, mean, M2), so it is as accurate as a one-pass Welford.

In [ ]:
var big = DaskArray.RandomNormal([4000, 2000], [1000, 500], seed: 11);
Console.WriteLine(big);
Console.WriteLine($"mean {big.Mean().Compute():F5}, std {big.Std().Compute():F5}");

var columnMeans = big.Mean(axis: 0).Compute();
Console.WriteLine($"column means: shape {string.Join('x', columnMeans.Shape.ToArray())}, max |mean| {columnMeans.ToArray().Max(Math.Abs):F4}");

var gramMatrix = (big.T.Dot(big) / 4000.0).Compute(new ThreadScheduler());
Console.WriteLine($"X^T X / n: diagonal ~ {Enumerable.Range(0, 2000).Average(i => gramMatrix[i, i]):F4}");

## DataFrame: lazy, optimised, partitioned (`dask.dataframe`)

Filters and projections are expressions, not lambdas, so the optimiser can read them: it pushes
predicates into the CSV scan and below joins, and reads only the columns that are used.

In [ ]:
var prices = DaskFrame.ReadCsv("../datasets/finance_timeseries.csv", blockSize: 8 * 1024);
var busyMonths = prices
    .WithColumn("month", Col("date").Month())
    .Filter(Col("volume") > 1_000_000)
    .GroupBy("ticker", "month").Agg(("close", "mean"), ("volume", "count"))
    .SortBy("volume_count", ascending: false)
    .Limit(5);

Console.WriteLine("Unoptimised:");
Console.WriteLine(busyMonths.Explain(optimized: false));
Console.WriteLine("Optimised:");
Console.WriteLine(busyMonths.Explain());
Console.WriteLine(busyMonths.Compute());

## SQL (`dask-sql`)

A query compiles to the same plan a DataFrame builds, so it gets the same optimiser and runs on
any scheduler.

In [ ]:
var sqlContext = new SqlContext().RegisterCsv("titanic", "../datasets/titanic.csv");
var byClass = sqlContext.Execute("""
    SELECT class, COUNT(*) AS passengers, ROUND(AVG(fare), 2) AS avg_fare, ROUND(AVG(survived) * 100, 1) AS survived_pct
    FROM titanic
    GROUP BY class
    ORDER BY avg_fare DESC
    """);
Console.WriteLine(byClass);

## A local cluster (`dask.distributed`)

Workers talk TCP and serialise every task, even in-process, so what runs here runs on a real
cluster. Only `[RemoteTask]` methods cross the wire, and every array, DataFrame and SQL kernel is
one. If a worker dies, the client recomputes what was lost from the graph's lineage.

In [ ]:
var cluster = await LocalCluster.StartAsync(workers: 3, threadsPerWorker: 2);
foreach (var w in await cluster.Client.GetWorkerInfoAsync())
    Console.WriteLine($"{w.Name} at {w.Address}, {w.Threads} threads");

var bySex = sqlContext.Sql("SELECT sex, ROUND(AVG(survived) * 100, 1) AS survived_pct FROM titanic GROUP BY sex").Compute(cluster.Client);
Console.WriteLine(bySex);
Console.WriteLine(cluster.Client.LastRun);

var onCluster = big.Std().Compute(cluster.Client);
Console.WriteLine($"std on the cluster {onCluster:F5}; tasks per worker: {string.Join(", ", cluster.Client.LastRun.TasksPerWorker.Select(kv => $"{kv.Key}={kv.Value}"))}");
await cluster.DisposeAsync();

---
Dibuat oleh Gravicode Studios, dipimpin oleh Kang Fadhil